# PDF Loading and Cleaning 

1. Why do we need:
In real-world GenAI applications, a lot of useful information is stored inside PDFs:

📚 Research papers
🏢 Company documents
📋 HR policies
📑 Financial reports
📜 Legal documents
📖 E-books
🧑‍💻 Technical documentation
🎓 Study materials
🏥 Medical documents
🛠️ Product manuals

An LLM cannot directly understand a PDF file as a knowledge source.

We first need to:

PDF
 ↓
Extract Text
 ↓
Clean Text
 ↓
Split into Chunks
 ↓
Create Embeddings
 ↓
Store in Vector Database
 ↓
Retrieve Relevant Content
 ↓
LLM
 ↓
Answer

This is one of the fundamental pipelines behind RAG (Retrieval-Augmented Generation).

In [1]:
# PDF loaders
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader

In [7]:
# PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("pdfs/fp_monitoring_2000_frep_09_en.pdf")

docs = loader.load()

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/FirstChar': 32, '/LastChar': 150, '/Widths': [600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 600], '/Encoding': '/WinAnsiEncoding', '/BaseFont': '/EPJGFD+Courier', '/FontDescriptor': IndirectObject(741, 0, 4585320768)}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type

In [8]:
print(type(docs))

<class 'list'>


In [10]:
# checking the number pages

print(f"Loader {len(docs)} pages")

Loader 239 pages


In [11]:
print(docs[0].page_content)

Agreement No. SI2.304597 
Hospital Data Project 
 
           
 
 
 
 
 
 
HOSPITAL DATA PROJECT 
 
FINAL REPORT 
 
June 2003


In [13]:
print(docs[1].page_content[:100])

Agreement No. SI2.304597 
Hospital Data Project 
 
 i
 
 
Acknowledgements 
 
 
On behalf of the Hos


In [14]:
print(docs[0].metadata)

{'producer': 'PDF CoDe 1.00.20030707 (c) 2002-2003 European Commission', 'creator': 'Dept of Health', 'creationdate': '2003-10-13T08:24:32+00:00', 'moddate': '2003-10-13T10:55:10+02:00', 'title': 'The Hospital Data Project', 'author': 'Hugh Magee', 'subject': 'Final report', 'keywords': 'European Commission, Public health, health monitoring, Hospital, Data, Project', 'format': 'Microsoft Word 10.0', 'source': 'pdfs/fp_monitoring_2000_frep_09_en.pdf', 'total_pages': 239, 'page': 0, 'page_label': '1'}


In [16]:
# PyMuPDFLoader

from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader('pdfs/fp_monitoring_2000_frep_09_en.pdf')

docs = loader.load()

In [17]:
# totoal length

print(f"Loader {len(docs)} pages")

Loader 239 pages


In [20]:
from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader


loader = DirectoryLoader(
    "pdfs",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
)

docs = loader.load()

print(f"total documents loaded : {len(docs)}")

total documents loaded : 254


# Why PDF Extraction Is Difficult

A PDF is not necessarily stored like this:

Hello World
This is a paragraph.

Internally, a PDF may store:

Individual text elements
Coordinates
Fonts
Images
Tables
Multiple columns
Headers
Footers
Page numbers

Therefore, extracted text can sometimes look messy.

In [21]:
raw_pdf_text = """Agreement No. SI2.304597 
Hospital Data Project 
 
           
 
 
 
 
 
 
HOSPITAL DATA PROJECT 
 
FINAL REPORT 
 
June 2003
"""

In [22]:
def clean_text(text):
    """
    Clean the text by removing special characters,
    numbersm and converting into lower case
    """

    text = " ".join(text.split())

        # Fix ligatures
    text = text.replace("ﬁ", "fi")
    text = text.replace("ﬂ", "fl")

    return text



In [23]:
text = " hello how are itk fam... "
clean_text(text)

'hello how are itk fam...'

In [25]:
text = "    hello how are itk fam...      " 
spl = text.split()
spl

['hello', 'how', 'are', 'itk', 'fam...']

In [26]:
# converted list to join back strings

text = " ".join(spl)

In [27]:
text

'hello how are itk fam...'

In [28]:
clean_text(raw_pdf_text)

'Agreement No. SI2.304597 Hospital Data Project HOSPITAL DATA PROJECT FINAL REPORT June 2003'

In [29]:
from ast import List

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List


class SmartPdfProcessor:
    """ 
    Adavnced PDF processing with error handling
    """
    def __init__(self,chunk_size=1000,chunk_overlap=200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            separators = [""] 
        )

    def _clean_text(self,text:str) -> str:
        """
        Clean the text by removing special characters,
        numbersm and converting into lower case
        """

        text = " ".join(text.split())

            # Fix ligatures
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")

        return text


    def process_pdf(self,pdf_path:str) -> List[Document]:
        """ Processing the pdf with smark chunk and metadata enrichment"""
        loader = PyMuPDFLoader(pdf_path)
        pages = loader.load()

        processed_chunks = []

        for page_num,page in enumerate(pages):
            ## clean text
            cleaned_text = self._clean_text(page.page_content)

            ## if i have empty pages skips
            if len(cleaned_text.strip()) <50:
                continue

            # create a text with enhanced metada:
            chunks = self.text_splitter.create_documents(
                texts = [cleaned_text],
                metadatas = [{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)
                }]
            )

            processed_chunks.extend(chunks)

        return processed_chunks




In [30]:
preprocesr = SmartPdfProcessor()

In [31]:
smart_chunk = preprocesr.process_pdf(pdf_path="pdfs/fp_monitoring_2000_frep_09_en.pdf")
print(f"Processed into {len(smart_chunk)} smartchunks")

if smart_chunk:
    print(" sample chaunks of metadata")
    for k,v in smart_chunk[0].metadata.items():
        print(f"{k} : {v}")

Processed into 631 smartchunks
 sample chaunks of metadata
producer : PDF CoDe 1.00.20030707 (c) 2002-2003 European Commission
creator : Dept of Health
creationdate : 2003-10-13T08:24:32+00:00
source : pdfs/fp_monitoring_2000_frep_09_en.pdf
file_path : pdfs/fp_monitoring_2000_frep_09_en.pdf
total_pages : 239
format : PDF 1.4
title : The Hospital Data Project
author : Hugh Magee
subject : Final report
keywords : European Commission, Public health, health monitoring, Hospital, Data, Project
moddate : 2003-10-13T10:55:10+02:00
trapped : 
modDate : D:20031013105510+02'00'
creationDate : D:20031013082432Z
page : 1
chunk_method : smart_pdf_processor
char_count : 91
